# SDXL + IP-Adapter Worker — Free-First Character-Consistent Stills

Google Colab T4 (16 GB VRAM) GPU worker for HomeBot & Ancient Pathways Movie Engine.
Processes deferred tickets (`ShotStatus.AWAITING_WORKER`) using Stable Diffusion XL + IP-Adapter to generate character-consistent stills at $0.00 spend.

## 1. Mount Google Drive
Mounts Google Drive where the project queue and tickets reside.
Expected directory: `MyDrive/Ancient_Pathways/movie_queue/tickets/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# Probe for queue directory in Drive (supports both HomeBot and Ancient Pathways)
DRIVE_BASE = Path('/content/drive/MyDrive')
POSSIBLE_QUEUES = [
    DRIVE_BASE / 'HomeBot' / 'movie_queue',
    DRIVE_BASE / 'Ancient_Pathways' / 'movie_queue',
]
DRIVE_ROOT = next((q for q in POSSIBLE_QUEUES if q.exists()), POSSIBLE_QUEUES[0])
TICKETS_DIR = DRIVE_ROOT / 'tickets'
INPUTS_DIR = DRIVE_ROOT / 'inputs'
OUTPUTS_DIR = DRIVE_ROOT / 'outputs'

TICKETS_DIR.mkdir(parents=True, exist_ok=True)
INPUTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Drive mounted. Queue directory: {DRIVE_ROOT}")
print(f"Tickets directory: {TICKETS_DIR}")

## 2. Install PyTorch, Diffusers, and IP-Adapter Dependencies
Installs diffusers, transformers, accelerate, insightface, and image processing tools.

In [ ]:
%pip install -q diffusers transformers accelerate safetensors insightface onnxruntime-gpu opencv-python pillow

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 3. Load SDXL + IP-Adapter on T4 (16 GB)
Loads `stabilityai/stable-diffusion-xl-base-1.0` in float16 with attention slicing, and attaches IP-Adapter.

In [ ]:
from diffusers import AutoPipelineForText2Image, DDIMScheduler
from PIL import Image

MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"
IP_ADAPTER_REPO = "h94/IP-Adapter"
IP_ADAPTER_SUBFOLDER = "sdxl_models"
IP_ADAPTER_WEIGHT = "ip-adapter-plus_sdxl_vit-h.safetensors"

print("Loading SDXL pipeline in float16...")
pipeline = AutoPipelineForText2Image.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
)

pipeline.scheduler = DDIMScheduler.from_config(pipeline.scheduler.config)

# Enable memory optimization for T4
pipeline.enable_attention_slicing()
pipeline.enable_model_cpu_offload()

print("Loading IP-Adapter weights...")
pipeline.load_ip_adapter(
    IP_ADAPTER_REPO,
    subfolder=IP_ADAPTER_SUBFOLDER,
    weight_name=IP_ADAPTER_WEIGHT,
)
pipeline.set_ip_adapter_scale(0.7)
print("SDXL + IP-Adapter ready on GPU!")

## 4. Ticket Queue Processor Function
Processes a single `ticket.json` file: loads character visual references, applies IP-Adapter image conditioning, generates the still, and updates `status.json` to `IMAGE_GENERATED`.

In [ ]:
import json
from datetime import datetime

def process_ticket(ticket_path: Path):
    with open(ticket_path, "r", encoding="utf-8") as f:
        ticket = json.load(f)

    shot_id = ticket.get("shotId", "unknown")
    prompt = ticket.get("prompt", "")
    negative_prompt = ticket.get("negativePrompt", "blurry, distorted, low quality, artifacts, bad anatomy, deformed")
    width = ticket.get("width", 1024)
    height = ticket.get("height", 576)
    char_refs = ticket.get("stagedCharacterRefs", ticket.get("characterRefs", []))
    seed = ticket.get("seed", 42)
    rel_out = ticket.get("relativeOutputPath")

    if rel_out:
        out_img_path = DRIVE_ROOT / rel_out
    else:
        shot_dir = Path(ticket.get("shotDir", str(ticket_path.parent)))
        out_img_path = shot_dir / "image" / f"{shot_id}.png"

    # Check if already generated (resuming after restart)
    if out_img_path.exists() and out_img_path.stat().st_size > 0:
        print(f"Output image already exists for {shot_id}: {out_img_path}")
        ticket["status"] = "IMAGE_GENERATED"
        ticket["completedAt"] = datetime.utcnow().isoformat() + "Z"
        ticket["outputFile"] = str(out_img_path)
        with open(ticket_path, "w", encoding="utf-8") as f:
            json.dump(ticket, f, indent=2)
        return out_img_path

    print(f"\n--- Processing {shot_id} ---")
    print(f"Prompt: {prompt[:80]}...")
    print(f"Resolution: {width}x{height}, Character refs: {len(char_refs)}")

    # Load reference images for IP-Adapter
    ref_images = []
    for ref_path in char_refs:
        p = Path(ref_path)
        if not p.is_absolute():
            p = DRIVE_ROOT / p
        if p.exists():
            ref_images.append(Image.open(p).convert("RGB"))
        else:
            print(f"Warning: reference image not found: {p}")

    ip_adapter_kwargs = {}
    if ref_images:
        ip_adapter_kwargs["ip_adapter_image"] = ref_images[0] if len(ref_images) == 1 else ref_images

    try:
        generator = torch.Generator("cuda").manual_seed(seed)
        image = pipeline(
            prompt=prompt,
            negative_prompt=negative_prompt,
            width=width,
            height=height,
            num_inference_steps=30,
            guidance_scale=7.5,
            generator=generator,
            **ip_adapter_kwargs,
        ).images[0]

        # Save output still image atomically
        out_img_path.parent.mkdir(parents=True, exist_ok=True)
        tmp_img_path = out_img_path.parent / f".tmp_{out_img_path.name}"
        image.save(tmp_img_path)
        tmp_img_path.replace(out_img_path)
        print(f"Saved still to: {out_img_path}")

        # Update ticket status
        ticket["status"] = "IMAGE_GENERATED"
        ticket["completedAt"] = datetime.utcnow().isoformat() + "Z"
        ticket["outputFile"] = str(out_img_path)
        ticket.pop("error", None)
        with open(ticket_path, "w", encoding="utf-8") as f:
            json.dump(ticket, f, indent=2)

        # Update local status.json if reachable
        shot_dir_str = ticket.get("shotDir")
        if shot_dir_str:
            status_path = Path(shot_dir_str) / "status.json"
            if status_path.exists():
                try:
                    with open(status_path, "r", encoding="utf-8") as f:
                        status_data = json.load(f)
                    status_data["status"] = "IMAGE_GENERATED"
                    status_data["updatedAt"] = datetime.utcnow().isoformat() + "Z"
                    with open(status_path, "w", encoding="utf-8") as f:
                        json.dump(status_data, f, indent=2)
                except Exception:
                    pass

        return out_img_path
    except Exception as e:
        print(f"Error generating {shot_id}: {e}")
        ticket["status"] = "FAILED"
        ticket["error"] = str(e)
        ticket["updatedAt"] = datetime.utcnow().isoformat() + "Z"
        with open(ticket_path, "w", encoding="utf-8") as f:
            json.dump(ticket, f, indent=2)
        return None

## 5. Run All Pending Tickets
Scans the tickets directory and processes all pending shots in batch.

In [ ]:
def process_all_pending_tickets():
    if not TICKETS_DIR.exists():
        print(f"Tickets directory does not exist: {TICKETS_DIR}")
        return

    tickets = sorted(list(TICKETS_DIR.glob("*.json")))
    print(f"Found {len(tickets)} tickets in {TICKETS_DIR}")
    processed_jobs = set()
    for t_path in tickets:
        try:
            with open(t_path, "r", encoding="utf-8") as f:
                t = json.load(f)
            job_id = t.get("jobId", t.get("ticketId", t_path.stem))
            if job_id in processed_jobs:
                continue
            if t.get("status") == "AWAITING_WORKER":
                processed_jobs.add(job_id)
                process_ticket(t_path)
        except Exception as e:
            print(f"Error processing {t_path}: {e}")

process_all_pending_tickets()

## 6. (Optional) Continuous Watcher Mode
Run this cell to keep the Colab runner active, checking for new tickets every 15 seconds.

In [ ]:
import time

WATCH_MODE = False  # Set to True to enable polling
POLL_INTERVAL_SEC = 15

print("Watcher mode. Set WATCH_MODE = True and run to poll continuously.")
while WATCH_MODE:
    try:
        process_all_pending_tickets()
        time.sleep(POLL_INTERVAL_SEC)
    except KeyboardInterrupt:
        print("Watcher stopped by user.")
        break
    except Exception as e:
        print(f"Watcher loop error: {e}")
        time.sleep(POLL_INTERVAL_SEC)